# Bubble Bi

### Reading the stock market like a language

## The problem

Every day, a stock leaves a trace: open, high, low, close, volume. Markets repeat certain moods — calm drift, panic, sharp reversal — but nobody labels them, and there is no dictionary of them.

**The idea:** let the machine build that dictionary itself.

We compress each stock-day into a single **token** — one word out of 512 that the machine invents on its own. A stock's history then becomes a sentence:

```
AAPL: ... 147  147  391  208  208  63  →  what comes next?
```

From there it is the same trick as ChatGPT: read the sentence, predict the next word.

**Three stages:**

| | | |
|---|---|---|
| 1. **Tokenizer** | turns each stock-day into one token | ✅ built |
| 2. **Predictor** | a GPT that predicts the next token | ✅ built |
| 3. **Trader** | acts on it: long / short / flat | ⬜ not built |

> ⚠️ **The one rule:** nothing may ever look into the future. Break it and the model looks brilliant while losing real money. A test enforces it.

Run the cells below in order.

---
## 1. Settings

Every knob is here — nothing hidden in a config file.

**The tokenizer looks at each day through two windows**, then merges what it sees into a single token:

```
  TS  — what THIS stock has been doing   (one stock, over time)   ┐
                                                                  ├──→  ONE token
  CS  — what the WHOLE MARKET was doing  (all stocks, on a day)   ┘
```

Each window has its own settings.

In [ ]:
SETTINGS = dict(

    # ── Which companies, and how far back ────────────────────────────────
    tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "V", "JNJ", "WMT",
               "PG", "HD", "BAC", "XOM", "CVX", "KO", "PEP", "DIS", "CSCO", "INTC",
               "VZ", "T", "MRK", "PFE", "ABT", "NKE", "MCD", "CAT", "BA", "IBM"],
    start   = "2010-01-01",

    # ⚠️ SIX KNOBS BELOW ARE SEARCHED (section 7): `ts.days`, `ts.vocabulary`,
    # `cs.days`, `cs.vocabulary`, `model_size`, `learning_rate`. Every one of them is
    # commented out here ON PURPOSE, and that is not decoration:
    #
    #   Type a number for one of them and YOURS WINS -- the tuned answer for that exact
    #   setting is silently discarded. `apply()` (section 7) will still tell you when
    #   that happens and name both numbers, but the honest default is to not throw the
    #   search's answer away in the first place. Leave these six commented out and, IF
    #   a `tuned.json` has ever been found and committed, you inherit its answer simply
    #   by cloning; uncomment one only once you have a specific reason to override that
    #   one number yourself.
    #
    # ⚠️ There is no `tuned.json` in THIS repo, and section 7's search must not be
    # turned on to make one -- see the warning at the top of section 7 and of
    # `bubble_bi/tuning.py`: the objective it searches for was REPUDIATED BY
    # MEASUREMENT (gameable, and it scores a standalone tokenizer this project no
    # longer trains). Until it is retargeted at the joint objective, these six simply
    # fall back to the plain defaults in `bubble_bi/settings.py` -- nothing breaks
    # either way.

    # ── ENTRY 1 — TS: what THIS stock has been doing ─────────────────────
    ts = dict(
        # days          = 15,    # days of the stock's own history it reads
        # vocabulary    = 512,   # size of its private dictionary
        encoder_depth = 3,
        decoder_depth = 2,
        batch         = 256,   # ~83,000 grids: one per company per day
    ),

    # ── ENTRY 2 — CS: what the WHOLE MARKET was doing ────────────────────
    cs = dict(
        # days          = 5,     # days of market-wide history it reads
        # vocabulary    = 512,
        encoder_depth = 3,
        decoder_depth = 2,
        batch         = 64,    # only ~2,800 grids — one per DAY, so a smaller batch
    ),

    # ── Where TS and CS meet, BEFORE either is quantised ──────────────────
    # ⚠️ There is no `vocabulary` here any more. TS and CS each keep their OWN codebook
    # (inside their own blocks above) -- there is no THIRD, fused codebook: that was the
    # one the predictor used to see, and the one we watched collapse to 10 words of 512.
    # `fusion` has no codebook knobs of its own to configure.
    fusion = dict(
        depth      = 2,
        attend_to  = "companies",  # what CS offers the cross-attention to choose between:
                                   #   "companies" → 30 keys (a bank can attend to banks)
                                   #   "cells"     → 150 keys (a peer on a specific day)
                                   #   "days"      → 5 keys.  ⚠️ NEVER pick this: at
                                   #                 cs["days"] == 1 it offers exactly ONE
                                   #                 key, and softmax over one key is
                                   #                 identically 1.0 -- every company would
                                   #                 get the SAME market vector and no
                                   #                 gradient could ever change that.
                                   #                 `check()` refuses the combination
                                   #                 outright; "companies" never has the
                                   #                 problem, so that is the default.
        batch      = 32,
    ),

    # ── The predictor: the GPT that reads the sentences ──────────────────
    predictor = dict(
        sentence_length = 64,  # how many past days it reads before guessing
        depth           = 4,
    ),

    # ── What the model is pulled towards, and how hard ───────────────────
    # These fight each other, and the balance IS the design.
    # (`commitment`/`diversity`/`decay` are CODEBOOK knobs, not loss weights. There are
    # TWO codebooks now -- ts and cs -- so each gets its own, inside its own block above,
    # rather than a shared triple sitting here.)
    loss = dict(
        naming  = 0.1,   # predict tomorrow's WORD.  ⚠️ the one that rewards CHEATING --
                         # making every day the same word scores perfectly on this alone.
        predict = 1.0,   # draw tomorrow's CANDLE.   the real objective -- the only
                         # target the model cannot manufacture for itself.
        recon   = 1.0,   # rebuild TODAY's own TS grid and CS grid.  THE ANCHOR. Every
                         # codebook in this project that had one stayed healthy; the one
                         # that did not (the old fused codebook) collapsed.
                         # ⚠️ STORM reports 1e-3 here -- DO NOT COPY IT. Their
                         # reconstruction is a SUM over millions of elements; ours is a
                         # `.mean()`. Copying the number would silently delete the anchor.
    ),

    # ── 🔎 THE SEARCH (section 7) — how those six knobs get found ────────
    #
    # ⚠️⚠️⚠️ DO NOT SET run = True. ⚠️⚠️⚠️ The objective this search optimises was
    # REPUDIATED BY MEASUREMENT: section 8 found it gameable (it trivially rewards
    # shrinking `days`, because what it asks the token to keep is already sitting in
    # its own input window), and it scores a STANDALONE, reconstruction-only tokenizer
    # that this project no longer trains at all -- see the warning at the top of
    # `bubble_bi/tuning.py`. Leave this OFF until it is retargeted at the JOINT
    # objective (perplexity and drawing-vs-shrugging under `train_joint`).
    #
    #     run = True    SEARCH for them.  GPU, ~15 min per model. Writes `tuned.json`.
    #     run = False   INHERIT them.     Reads `tuned.json` if it exists. Seconds, no GPU.
    #
    # There is no `tuned.json` committed to this repo today. If the search is ever
    # retargeted and re-run, its answer would be committed as `tuned.json`, and
    # everyone after that inherits it simply by cloning — nobody pays for it twice.
    search = dict(
        run    = False,
        trials = 12,   # ⚠️ 12 is a SCREEN, not an optimum. It will catch a badly-wrong
                       #    setting; it will NOT find the best one. Raise it to 60+ for a
                       #    real search — the code is identical, it just gets better.
        steps  = 600,  # per trial. A CEILING, not a target: early stopping usually ends
                       #    a trial well before this.
    ),

    # ── Shared ───────────────────────────────────────────────────────────
    # model_size is searched too -- see the note above `ts`, and leave it commented
    # out to inherit the tuning.
    # model_size = 128,   # brain width. Shared on purpose: TS and CS meet in a
    #                     # cross-attention layer, which needs both the same width.
    #                     # ⬆️ The STORM paper uses 256. Try that on a GPU.

    # ── Training ─────────────────────────────────────────────────────────
    steps         = 2000,   # ⬆️ ON A GPU, RAISE THIS. Try 5,000+.
                            #
                            # ⚠️ It must stay ABOVE `search["steps"]` (600) or the search
                            # refuses to run — and rightly. The search trains each trial
                            # for a short SPRINT, then confirm() re-trains the winner at
                            # THIS budget to check it still wins when it actually counts.
                            # If this were the smaller of the two, the guard would be
                            # re-running the sprint SHORTER than the sprint, which is not
                            # a guard at all. It shipped at 300 against a 600-step sprint,
                            # and ran backwards for two whole GPU runs.
                            #
                            # (CS keeps its own budget — `cs["steps"]` — because it has
                            # ~30x less data and needs the passes. confirm() honours it.)
    # learning_rate is searched too -- see the note above `ts`, and leave it commented
    # out to inherit the tuning.
    # learning_rate = 1e-4,
    seed          = 42,     # feeds the hyperparameter search (section 7) -- same seed,
                            # same trials, every time -- AND every training run within
                            # it: `_run_one` reseeds torch from this exact number before
                            # building each model, so two trials given the same config
                            # get the same starting weights, and the confirm() step
                            # (which trains the winner and the incumbent back to back)
                            # compares them on the config alone, not on who happened to
                            # get the luckier initialisation.

    # ── Where things are kept ────────────────────────────────────────────
    # ⬆️ ON COLAB, point this at your Drive so a disconnect cannot eat your work:
    #        data_dir = "/content/drive/MyDrive/bubble_bi"
    data_dir = "artifacts",
)

---
## 2. Get the code

Downloads the project and installs what it needs. Safe to re-run — it skips anything already there.

> ### 🔺 Running this on Colab?
>
> **1. Turn on the GPU — and then *restart*.**
> *Runtime → Change runtime type → **T4 GPU**,* then *Runtime → **Restart session***.
>
> **The restart is not optional, and skipping it is the single most common way to get stuck here.** Colab's CPU runtime ships a PyTorch that cannot use a GPU *at all*. A kernel that is already running keeps the PyTorch it started with — so you can tick "T4 GPU", watch nothing change, and be left staring at `Hardware: CPU` with no idea why. The check below will tell you plainly if this has happened.
>
> On a CPU this notebook is a toy. On a GPU it is the real thing.
>
> **2. Mount your Drive**, and point `data_dir` at it in the settings above:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> # then in SETTINGS:  data_dir = "/content/drive/MyDrive/bubble_bi"
> ```
> **Colab disconnects.** It will disconnect halfway through a long run and it does not care. Every trained model is written to `data_dir` the moment it finishes, and re-running this notebook **loads what is already there instead of training it again**. Point it at Drive and a dropped session costs you nothing.
>
> **3. Raise `steps`.** 300 is a laptop run. Nothing it produces should be believed.

In [ ]:
import importlib, os, subprocess, sys

OWNER, REPO, BRANCH = "hockper", "Quant-AI-2026", "main"
FOLDER = "bubble-bi"
URL = f"https://github.com/{OWNER}/{REPO}.git"

_token = None          # held in memory for this session only. Never written anywhere.


def run(*command, secret=None):
    """Run a command and, if it fails, SHOW WHY — with any token scrubbed out."""
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode:
        said, shown = (done.stderr.strip() or done.stdout.strip()), " ".join(command)
        if secret:                    # never let a token reach the screen or the notebook
            said, shown = said.replace(secret, "***"), shown.replace(secret, "***")
        raise RuntimeError(f"`{shown}` failed:\n\n{said}")
    return done


def _needs_login(why) -> bool:
    return "could not read Username" in str(why) or "Authentication" in str(why)


def _with_token():
    """The repo URL with a token in it. Asked for once, kept in memory, never saved."""
    global _token
    if not _token:
        try:
            from google.colab import userdata     # Colab → 🔑 Secrets → GITHUB_TOKEN
            _token = userdata.get("GITHUB_TOKEN")
            print("🔑 using GITHUB_TOKEN from your Colab secrets")
        except Exception:
            pass
    if not _token:
        from getpass import getpass
        print("This repo is private. Paste a GitHub token with `repo` access.")
        print("(Hidden as you type, never printed, never saved into this notebook.)")
        print("Nicer: put it in Colab's 🔑 Secrets panel as GITHUB_TOKEN and re-run.")
        _token = getpass("GitHub token: ").strip()
    return f"https://{_token}@github.com/{OWNER}/{REPO}.git", _token


def git(*command):
    """Run a git command that talks to GitHub — without a token if we can, with one if
    we must. The token is passed on the command line and never stored on disk."""
    try:
        return run("git", *command, URL, BRANCH)
    except RuntimeError as why:
        if not _needs_login(why):
            raise
    url, token = _with_token()
    return run("git", *command, url, BRANCH, secret=token)


def torch_now():
    """Which PyTorch is installed right now — or None if there isn't one."""
    try:
        import torch
        return f"{torch.__version__} (cuda {torch.version.cuda})"
    except ImportError:
        return None


# --- get the code, or bring in whatever has been pushed since -----------------
if not os.path.exists("bubble_bi") and os.path.isdir(FOLDER):
    os.chdir(FOLDER)                       # on disk, but the kernel restarted outside it

if os.path.exists("bubble_bi"):            # we are inside the repo -> UPDATE it
    was = run("git", "rev-parse", "--short", "HEAD").stdout.strip()
    git("pull", "--ff-only")
    now = run("git", "rev-parse", "--short", "HEAD").stdout.strip()
    print(f"↻  pulled new code: {was} → {now}" if was != now
          else f"✓  already up to date ({now})")
else:                                      # nothing here -> CLONE it
    try:
        run("git", "clone", "-b", BRANCH, URL, FOLDER)
    except RuntimeError as why:
        if not _needs_login(why):
            raise
        url, token = _with_token()
        run("git", "clone", "-b", BRANCH, url, FOLDER, secret=token)
    # Scrub any token back out of .git/config — otherwise it sits there in plain text.
    subprocess.run(["git", "-C", FOLDER, "remote", "set-url", "origin", URL], check=True)
    os.chdir(FOLDER)
    print(f"✓  cloned {OWNER}/{REPO}")

# --- install what we need, and WATCH what it does to PyTorch ------------------
# ⚠️ `requirements.txt` deliberately does not list torch: Colab already ships one built
# for its GPU, and replacing it with a CPU-only wheel would leave the GPU sitting idle
# while everything still "works". That comment has been in the file all along — but the
# install ran quietly, so if pip ever HAD swapped torch out, nothing would have said so
# and we would be left staring at "Hardware: CPU" with no idea why. So: look before,
# look after, and shout if it moved.
torch_before = torch_now()
pip = run(sys.executable, "-m", "pip", "install", "-r", "requirements.txt")
torch_after = torch_now()

changed = [line for line in pip.stdout.splitlines()
           if line.startswith(("Installing collected packages", "Successfully installed"))]
print("\n".join(changed) if changed else "✓  dependencies already satisfied")

if torch_before != torch_after:
    print(f"\n🚨  pip CHANGED PyTorch:  {torch_before}  →  {torch_after}")
    print("    Nothing here is allowed to do that. If the new one says `cuda None`, the")
    print("    GPU is now unreachable: Runtime → Disconnect and delete runtime, start again.")

sys.path.insert(0, os.getcwd())

# Python caches imported code. Without this, pulling new code would change the files on
# disk while the notebook happily kept running the OLD copy still sitting in memory.
for loaded in [m for m in sys.modules if m == "bubble_bi" or m.startswith("bubble_bi.")]:
    del sys.modules[loaded]
importlib.invalidate_caches()

import bubble_bi as bb

settings = bb.check(SETTINGS)     # fills in defaults, catches typos
print()
print(bb.summary(settings))

### ✅ Check

Every section of this notebook ends like this: it *proves* the step worked, then tells you what you now have. If something is wrong it stops here rather than letting you carry on.

In [ ]:
bb.verify.setup(settings)

---
## 3. Get the prices

Download every company's daily history and put it in one table.

Each row is **one company on one day**, holding the six numbers a trading day leaves behind — plus `target`, tomorrow's return.

> `target` is *the answer*. It is the only column allowed to look forward, and it is never given to a model as an input.

Downloads are cached, so this is slow once and instant afterwards.

In [ ]:
prices = bb.data.download(settings)

prices.head()

In [ ]:
bb.verify.prices(prices, settings)

---
## 4. Describe each day properly

Raw prices are a thin description of a trading day. Before the machine can find *moods*, we re-describe every day from five angles:

| | |
|---|---|
| **price** | where it is going — returns, moving averages, RSI, MACD |
| **volatility** | how violently it moves — four estimators, each reading a different part of the candle |
| **memory** | whether the past still echoes — Hurst, fractional differencing, entropy |
| **microstructure** | what it *costs* to trade — hidden spread, illiquidity |
| **flow** | who is pushing, and how hard — volume dynamics |

That's **26 numbers per company per day**. This table is what the tokenizer will learn to squeeze into single tokens.

In [ ]:
data = bb.data.add_features(prices, settings)

data.tail()

### ✅ Check

The important one. **We do not trust that our features are backward-looking — we prove it.**

The check below takes the data, **deletes the future**, recomputes every feature from scratch, and verifies that not a single past value changed. If any feature were secretly peeking at tomorrow, deleting tomorrow would change it, and this catches it.

This is the check that separates a real trading model from one that fools its author.

In [ ]:
bb.verify.features(data, prices, settings)

---
## 5. Build the two entries

Time for the machine that invents the dictionary.

**TS and CS look like two different problems. They aren't.** They are the same problem at two sizes — so they are the *same class*, configured differently:

```
TS     1 company  × 15 days × 26 features   →  one word     "what THIS stock is doing"
CS    30 companies × 5 days × 26 features  →  one word     "what the MARKET is doing"
```

TS is simply CS watching a single company. Each one works the same way:

| | |
|---|---|
| **encoder** | reads the whole grid and boils it down to a single vector |
| **codebook** | snaps that vector to the **nearest word** in its dictionary of 512 |
| **decoder** | tries to rebuild the *entire grid* from that one word |

Training makes the rebuild as accurate as possible. **That is the whole trick.** Nobody tells the machine what the words should mean — it is simply forced to describe a grid of market data using one word out of 512, and the only way to do that well is for the words to come to mean something.

In [ ]:
n_features  = len(bb.data.names())        # 26
n_companies = len(settings["tickers"])    # 30

# The SAME class, twice. Only the numbers differ.
ts = bb.models.VQVAE(companies=1,           features=n_features,
                     width=settings["model_size"], **settings["ts"])

cs = bb.models.VQVAE(companies=n_companies, features=n_features,
                     width=settings["model_size"], **settings["cs"])

print("TS :", ts.describe())
print("CS :", cs.describe())

### ✅ Check

Push a fake grid through both machines and watch what comes out. In particular: **gradients must survive the snap.**

Snapping a vector to its nearest word is a step function — it has no slope, so ordinarily no learning signal could pass back through it. A trick called *straight-through* smuggles the gradient past. If that were broken, training would appear to run, the loss would sit still, and nothing would tell you why.

In [ ]:
bb.verify.models(ts, cs, settings)

---
## 6. Cut the table into grids

The models eat grids, not tables. Cut them out, normalise them, and divide history into three periods:

| | |
|---|---|
| **learn** | the model trains on these days |
| **tune** | used to watch progress and stop at the right moment |
| **test** | untouched until the very end. The only honest score. |

**The split is strictly by date.** Learn on the past, tested on the future — the way it would actually have to work.

### Every company is normalised against *itself*

A \$500 stock and a \$20 stock are not comparable. Nor is a heavily-traded giant and a thin one. Left alone, **half of some features is nothing but *which company it is***:

```
close_frac   68% ← just the price level
amihud       41% ← just how liquid it is
obv_frac     39% ← just how many shares it trades
```

The codebook only has 512 words. If we hand it that, it will spend them memorising *"this is NVDA"* instead of *"this is a panic"*.

So each company is normalised **against its own history, down the time axis** — never against the other companies. Afterwards every feature says the same thing for everyone:

> *"How unusual is today — **for this company**?"*

⚠️ **What this deliberately throws away:** that one company is *genuinely* more volatile, or less liquid, than another. A sleepy utility at its own average volatility and a wild biotech at *its* own average volatility now look identical. That is the price of a word meaning the same thing whoever it describes.

> ### ⚠️ Three ways a trading model quietly cheats
>
> **1. Shuffling time.** Split days at random and the model trains on Thursday, then gets tested on Wednesday. We split by *date*.
>
> **2. Scaling with the future.** Compute that normalisation from *all* of history and the average of days you haven't seen yet leaks backwards into training. We measure it on the **learn period only**.
>
> **3. A target that reaches over the border.** The last learn day's target is *tomorrow's* return — and tomorrow is the first tune day. We drop that day. One row, and it's the difference between an honest boundary and a leaky one.
>
> The check below **proves** all of it, rather than promising it.

In [ ]:
batches = bb.data.make_tensors(data, settings)

batches.sizes()

In [ ]:
bb.verify.tensors(batches, ts, cs, settings)

---
## 7. Getting the settings right

> ### 🚫 ⚠️ DO NOT set `SETTINGS["search"]["run"] = True` — read this first
>
> The objective this section searches for was **repudiated by measurement**, in this
> branch's own section 8 findings: it turned out to be **gameable** (the search would
> trivially maximise it by shrinking `ts.days`/`cs.days` to the floor, because what it
> asks the token to preserve is already sitting in its own input window), and it scores
> a **standalone, reconstruction-only tokenizer with no fusion** — a model this project
> no longer trains at all. `train_joint` (section 8) trains everything together, against
> tomorrow's candle, instead.
>
> The machinery below — the two-stage search, the transfer guard, the boundary check —
> is sound and will be reused once it is retargeted at the **joint** objective. Until
> then, leave `search["run"] = False`.

Every number in `SETTINGS` above was a guess. `learning_rate` is `1e-4` because somebody
typed `1e-4`.

So we measure instead. But **not** by asking which settings rebuild the window best — we
already know that lies to us: handed the candle explicitly, the best compressor threw it
away. And **not** by asking which settings predict tomorrow — TS and CS are autoencoders,
they represent the *present*, and scoring them on a forecast would just rank noise.

We ask the only honest question: **does today survive the bottleneck?** A held-out linear
probe reads the token and tries to recover today's direction and today's volatility. That
is what the token is *for*, and whatever it loses here is lost forever — no predictor
downstream can recover information the tokenizer already threw away.

⚠️ **That question turned out not to be honest enough** — see the warning above. This
section is **off by default**, and there is no `tuned.json` committed to this repo today:
`apply()` below runs on plain defaults. If the search is ever retargeted at the joint
objective and re-run, its answer would be committed as `tuned.json`, and you would
inherit it simply by cloning.

In [ ]:
import pandas as pd

if settings["search"]["run"]:
    ts_best, ts_trials = bb.tuning.search("ts", batches, settings)
    cs_best, cs_trials = bb.tuning.search("cs", batches, settings)

    ts_verdict = bb.tuning.confirm("ts", ts_best, batches, settings)
    cs_verdict = bb.tuning.confirm("cs", cs_best, batches, settings)

    print(f"TS  head-to-head at full budget: winner {ts_verdict['winner']:+.3f} "
          f"vs incumbent {ts_verdict['incumbent']:+.3f} → kept the {ts_verdict['kept']}")
    print(f"CS  head-to-head at full budget: winner {cs_verdict['winner']:+.3f} "
          f"vs incumbent {cs_verdict['incumbent']:+.3f} → kept the {cs_verdict['kept']}")

    # ⚠️ DID THE WINNER LAND ON THE EDGE OF THE BOX WE GAVE IT?
    #
    # If it did, the real best value is OUTSIDE that box and no number of trials inside it
    # will ever find it — you would just keep re-running a search that cannot reach the
    # answer. Three GPU runs did exactly that: a 60-trial run pinned SIX of its twelve
    # winning values to a boundary (TS commitment 1.99 of max 2.0, CS vocabulary at the
    # smallest offered, CS model_size at the largest...) and nothing said a word.
    for label, verdict in (("TS", ts_verdict), ("CS", cs_verdict)):
        for line in bb.tuning.boundaries(label.lower(), verdict["settings"]):
            print("\n" + line)

    # ⚠️ A collapsed confirm() looks, at a glance, like an ordinary result -- `kept` still
    # says "incumbent", exactly as it would if the incumbent had genuinely won a fair
    # fight. It is not the same thing. `-inf` on BOTH sides means every configuration this
    # search tried -- the winner AND the incumbent -- destroyed the codebook, so there was
    # NOTHING to compare, and "kept the incumbent" only means it was kept by default, not
    # because the data said so. Say that plainly, and name the two knobs most likely
    # responsible: see `Codebook._crowding` and `score_tokenizer`'s own docstring for why
    # `commitment` too high pins the encoder to words it already has, and `diversity` too
    # low stops fighting the handful of words that win early.
    for label, verdict in (("TS", ts_verdict), ("CS", cs_verdict)):
        if bb.tuning.confirm_collapsed(verdict):
            print(f"\n⚠️  {label}: EVERY configuration destroyed the codebook this run -- "
                  f"the winner AND the incumbent both collapsed. There is no real result "
                  f"to report: '{verdict['kept']}' was kept only because there was nothing "
                  f"left to compare it against, not because it won anything.\n"
                  f"    `commitment` too high, or `diversity` too low, is what kills a "
                  f"codebook. Widen the search or hand-pick a gentler balance and try "
                  f"again.")

    # ⚠️ The score recorded here must describe `ts_verdict["settings"]` -- the config
    # that was ACTUALLY saved just above -- not the best row of `ts_trials`. `confirm()`
    # can (and sometimes does) keep the INCUMBENT over the search's own winner; when it
    # does, `ts_trials.iloc[0]` is the score of a config that was just thrown out. It is
    # also the wrong SHAPE: `iloc[0]` is one stage's row, so the other stage's own knobs
    # come back as `NaN` in it, and `json.dumps` writes a bare `NaN` literal -- which
    # `json.loads` in Python happily reads back (masking the bug) but no other JSON
    # parser will accept. `confirm()`'s own return -- kept/winner/incumbent, three plain
    # floats -- is what was actually decided, so that is what gets saved. And when EVERY
    # trial collapsed (both `-inf`, printed above), `save()` itself turns that into JSON
    # `null` plus an explicit `collapsed: true` flag, rather than writing a bare
    # `-Infinity` no other parser can read -- see `bb.tuning.save`'s own docstring.
    bb.tuning.save({"ts": ts_verdict["settings"], "cs": cs_verdict["settings"],
                    "score": {"ts": {"kept": ts_verdict["kept"],
                                     "winner": ts_verdict["winner"],
                                     "incumbent": ts_verdict["incumbent"]},
                              "cs": {"kept": cs_verdict["kept"],
                                     "winner": cs_verdict["winner"],
                                     "incumbent": cs_verdict["incumbent"]}}}, settings)
    # `search()` now stamps its own `entry` column on every row (used to be the
    # notebook's job, via `.assign(entry=...)` -- which meant the real
    # `search() → disagreements()` composition was never actually exercised by a test).
    # `ignore_index=True` still matters here: `ts_trials` and `cs_trials` both come back
    # from `search()` indexed 0..n-1 (sorted, never re-indexed), so concatenating them
    # WITHOUT this gives the combined table duplicate row labels -- TS's row 0 and CS's
    # row 0 sitting right on top of each other. `bb.tuning.disagreements()` below resets
    # the index defensively too, but there is no reason to hand it a table with
    # colliding labels in the first place.
    trials = pd.concat([ts_trials, cs_trials], ignore_index=True)
else:
    trials = None

# `ts`, `cs` and `batches` above were built from THIS -- keep it, so the next cell can
# tell whether `apply()` actually changed anything, rather than rebuilding blind.
before = settings

# ⚠️ NEVER write back into SETTINGS. It is what the HUMAN typed, and precedence is
# DEFAULTS < tuned.json < TYPED. This line used to read `SETTINGS, note = apply(SETTINGS)`
# -- reassigning the very dict it feeds back in -- so on the NEXT run the PREVIOUS run's
# tuned values were sitting in SETTINGS, `apply()` read them as things you had typed, and
# typed beats tuned. The new tuning was silently thrown away.
#
# It really happened: a 60-trial search found ts.days=5 / model_size=128, wrote them to
# tuned.json, and the notebook then trained days=10 / width=64 -- the answer from the run
# BEFORE. A search whose own output overrides its next output can only ever be run once.
merged, note = bb.tuning.apply(SETTINGS)      # SETTINGS itself is left exactly as you wrote it
settings = bb.check(merged)
print(note)

### Rebuild what the tuning just changed

`ts`, `cs` and `batches` were all built **earlier** in this notebook (sections 5 and
6), from whatever `SETTINGS` said before the tuning above ever ran. If `apply()` just
changed a setting — a tuned `vocabulary`, a better `commitment`, a different `days` —
those three objects are now **stale**: built from numbers nobody is using any more.
Training them next would train the *old* settings under a *new* name, and every
number this section just spent effort finding would silently do nothing.

So: rebuild `ts` and `cs` from the settled settings. That costs nothing — they are
freshly-initialised machines, not trained ones. `batches` is not free to rebuild on
the real 30-company panel, so it is only rebuilt when `days` actually changed the
*shape* of a grid — and either way, the notebook says out loud which one it did,
rather than deciding silently.

In [ ]:
# `ts`/`cs` are cheap to rebuild -- they are empty, freshly-initialised machines, not
# trained ones -- so that always happens, unconditionally. This is the fix for the bug
# that made the whole tuning section inert: without it, section 8 below would go on to
# train THESE two objects, built from the settings that existed before `apply()` ran.
ts = bb.models.VQVAE(companies=1,           features=n_features,
                     width=settings["model_size"], **settings["ts"])
cs = bb.models.VQVAE(companies=n_companies, features=n_features,
                     width=settings["model_size"], **settings["cs"])
print("TS :", ts.describe())
print("CS :", cs.describe())

# `batches` is the expensive one on a real panel, so only rebuild it when `days` --
# the one setting that changes the SHAPE of a grid -- actually moved. Either way, say
# so: a silent skip here is exactly the kind of thing that goes unnoticed for months.
grids_changed = (before["ts"]["days"] != settings["ts"]["days"]
                 or before["cs"]["days"] != settings["cs"]["days"])
if grids_changed:
    print("\n`days` changed under tuning -- rebuilding `batches` to match the new grid shape.")
    batches = bb.data.make_tensors(data, settings)
else:
    print("\n`days` unchanged -- `batches` already matches the settled settings, skipping "
          "the rebuild.")

In [ ]:
if trials is not None:
    # `direction_se` is the NOISE FLOOR on `direction` -- how much of it is measurement
    # wobble rather than signal. It is not decoration: on the first real GPU run TS scored
    # direction 0.030 and CS scored 0.989, and without this column you cannot tell that the
    # first is at the edge of its own noise (nil) while the second is TWENTY-ONE standard
    # errors clear of it (overwhelming). A score without its error bar invites exactly the
    # kind of quoting this project keeps having to walk back.
    #
    # `before_quant` is the same probe run on the CONTINUOUS vector the token was quantised
    # FROM. It is often NaN on purpose -- see `_score_and_floor`: CS has ONE grid per DAY
    # (~600 tune rows), and a 256-wide dense probe fitted on ~420 of them has as many free
    # parameters as data. It scored an IMPOSSIBLE -4.164 while the token it was quantised
    # from scored +0.561, and quantising cannot ADD information. So it now refuses to answer
    # rather than print a number nobody should trust. NaN here means "not enough rows", not
    # "the model failed".
    # ⚠️ SHOW THE KNOBS. A trial without its configuration is uninterpretable -- you can
    # see that a row scored 0.77 and learn nothing about WHY, or test any hypothesis about
    # it. (Concretely: "did CS only keep today's direction because it was handed FEWER
    # DAYS?" is a real, sharp question, and it is unanswerable without the `days` column
    # sitting right next to `direction`.)
    knobs = [k for stage in bb.tuning.SPACE.values() for k in stage]
    shown = (["entry", "stage"] + [k for k in knobs if k in trials.columns]
             + ["score", "direction", "direction_se", "volatility", "before_quant",
                "words_used"])
    display(trials[[c for c in shown if c in trials.columns]].round(4))
    bb.plots.tuning_importance(trials)

    # TS and CS are two SEPARATELY-tuned models, so "the best trial across both" isn't
    # a real question -- `bb.tuning.disagreements()` checks each one on its own: is its
    # top-scoring trial also the one that kept the most DIRECTION? Direction is the
    # scarce quantity in this whole project, so if not, that's a genuine choice for you
    # to make, not a detail to bury -- see the function's own docstring for why this
    # lived, untested, straight in the notebook before, and how a duplicate-index bug
    # let it silently never fire even when it was true.
    for line in bb.tuning.disagreements(trials):
        print("\n" + line)

    # THE QUESTION THE WHOLE SEARCH EXISTS TO ANSWER. Did today's DIRECTION survive the
    # bottleneck -- or is it destroyed at the tokenizer, where no predictor downstream can
    # ever get it back? Read it against its own noise, never on its own.
    print("\n" + "─" * 68)
    for entry, rows in trials.groupby("entry"):
        best = rows.loc[rows["direction"].idxmax()] if rows["direction"].notna().any() else None
        if best is None:
            continue
        se = best.get("direction_se", float("nan"))
        clear = best["direction"] / se if se and se == se and se > 0 else float("nan")
        verdict = ("SURVIVES — the token carries it" if clear >= 5
                   else "at the noise floor — the token destroyed it")
        print(f"  {entry.upper()}  best direction {best['direction']:+.3f} "
              f"± {se:.3f}  ({clear:.0f}× its noise)   → {verdict}")
    print("─" * 68)

    n = settings["search"]["trials"]
    distinct = trials.drop_duplicates(subset=[k for k in knobs if k in trials.columns])
    if n < 40:
        print(f"\n⚠️  {n} trials is a SCREEN, not an optimum. It will catch a badly-wrong "
              f"setting.\n    It will NOT find the best one. Raise "
              f"SETTINGS['search']['trials'] to 60+\n    for a real search — the code is "
              f"identical, it just gets better.")
    print(f"\n    {len(distinct)} DISTINCT configurations out of {len(trials)} trials. "
          f"TPE converges and\n    re-suggests the same corner of a discrete space; a "
          f"repeat now costs nothing\n    (the score is looked up, not retrained), but it "
          f"also buys nothing. If these\n    two numbers are far apart, the space is too "
          f"small for the budget.")

In [ ]:
import inspect

tuned = bb.tuning.TUNED.exists()
reaches = set(inspect.signature(bb.models.VQVAE.__init__).parameters)
decorative = set(settings["ts"]) - reaches

# The bug this whole section exists to catch: `apply()` can settle on a tuned
# `vocabulary`/`days`/`commitment`/`diversity`/`decay`/`model_size` and the notebook
# can still go on to train the OLD `ts`/`cs`, built before the tuning ever ran -- the
# tuning would then be silently inert. `model_size` matters MOST of all: it is one of
# the SIX knobs the search actually tunes (`bb.tuning.SPACE["sizes"]`), and it is
# trivially observable on the model -- `ts.read.out_features` is the width of the
# model's very first layer -- so there is no excuse for this check to have missed it.
# `encoder_depth` is checked too, the same way (`ts.encoder.num_layers`): it is not
# itself searched, but it is exactly as observable as `decay`, which was already here.
# This does not ask "did we call the rebuild cell" (that is what a comment would
# assert, and comments rot) -- it reads the numbers straight back off the actual model
# objects and demands they match what `settings` says right now.
model_matches_settings = (
    ts.codebook.words == settings["ts"]["vocabulary"] and ts.days == settings["ts"]["days"]
    and ts.codebook.commitment == settings["ts"]["commitment"]
    and ts.codebook.diversity == settings["ts"]["diversity"]
    and ts.codebook.decay == settings["ts"]["decay"]
    and ts.read.out_features == settings["model_size"]
    and ts.encoder.num_layers == settings["ts"]["encoder_depth"]
    and cs.codebook.words == settings["cs"]["vocabulary"] and cs.days == settings["cs"]["days"]
    and cs.codebook.commitment == settings["cs"]["commitment"]
    and cs.codebook.diversity == settings["cs"]["diversity"]
    and cs.codebook.decay == settings["cs"]["decay"]
    and cs.read.out_features == settings["model_size"]
    and cs.encoder.num_layers == settings["cs"]["encoder_depth"]
)

bb.report(
    "7. The settings",
    [
        ("Every setting reaches a model", not decorative,
         "none are decorative" if not decorative else f"IGNORED: {sorted(decorative)}"),
        ("Commitment is the literature value",
         settings["ts"]["commitment"] <= 0.5,
         f"{settings['ts']['commitment']} (we ran at 1.0 by accident for months)"),
        ("A tuning exists to inherit", tuned,
         "tuned.json" if tuned else "none yet — running on defaults"),
        ("The models match the settings", model_matches_settings,
         f"TS {ts.codebook.words} words / {ts.days} days / width {ts.read.out_features}, "
         f"CS {cs.codebook.words} words / {cs.days} days / width {cs.read.out_features}"),
    ],
    have=f"""
    TS  {settings['ts']['vocabulary']} words, width {settings['model_size']},
        {settings['ts']['days']} days back, lr {settings['learning_rate']:.1e}
    CS  {settings['cs']['vocabulary']} words, {settings['cs']['days']} days back
    The search is OFF by default — the next person inherits tuned.json by cloning.
    """,
    known_problem=(
        None if tuned else
        "No tuning has been run yet. The settings are guesses — good ones, but guesses. "
        "Set SETTINGS['search']['run'] = True on a GPU to find better."
    ),
)

---
## 8. Train everything at once

The tokenizer used to be trained to **rebuild the present**, then frozen, and only then did
the predictor get a say. But rebuilding the present and carrying the future are different
jobs, and we measured the gap three ways — most damningly, our own tuning objective turned
out to be **gameable**: "does today survive the bottleneck?" is trivially maximised by making
the window *be* today, because what we asked the token to preserve was already sitting in its
input. The search found the loophole and drove `days` to the floor.

**Tomorrow is not in the grid.** A token cannot manufacture it. So everything now trains
against it — the encoders, both codebooks, the fusion and the GPT — under one loss.

### Two words a day, and both are anchored

| | |
|---|---|
| `ts_token` | what **this stock** did, *given what the market was doing* |
| `cs_token` | what the **market** did |

Each keeps its own dictionary **and its own decoder**, and must rebuild its own grid. That
reconstruction is the **anchor**: every dictionary in this project that had one stayed
healthy, and the one that did not — the old fused codebook — collapsed to **10 words out of
512**.

### ⚠️ The number to watch is perplexity, from the first line

This model **invents its own vocabulary and is then graded on predicting it.** "Make every
day the same word" scores perfectly. We have watched it happen: **92% accuracy at perplexity
2.2.**

The naming head now reads a **detached** copy of the words — it can learn to *read* the
language, but it can never rewrite the language to be easier. If perplexity still slides,
that protection has failed, and **the loss curve will look perfectly healthy while it does.**

> Judge it against its floors, never against zero:
> **persistence** ("tomorrow's word is today's word") and **shrugging** ("draw the average candle").

In [ ]:
# `ts`/`cs` are NOT rebuilt here -- they already are the settled-settings pair built
# (and section 7's own check just verified) in the "Rebuild what the tuning just
# changed" cell above. Rebuilding them again here used to mean section 7's verify
# checked ONE pair of freshly-initialised weights while THIS cell trained a SECOND,
# different pair -- "The models match the settings" was guarding objects that were
# about to be thrown away, not the ones that actually got trained.
tokenizer = bb.models.Tokenizer(ts, cs, model_size=settings["model_size"],
                                **settings["fusion"])
world = bb.models.WorldModel(
    tokenizer,
    sentence=settings["predictor"]["sentence_length"],
    depth=settings["predictor"]["depth"],
    **settings["loss"],
)
book = bb.data.make_sentences(batches, settings)

print(world.describe())
print(f"   {len(book['learn'].dataset):,} sentences to learn from")

world_history = None
if bb.keep.load(world, "world", settings) is None:
    world_history = bb.train_joint(world, book, settings)
    bb.keep.save(world, "world", settings, steps=settings["steps"])

# Re-run this cell and it loads instead of retraining. To force a fresh run:
#     bb.keep.forget("world", settings)

In [ ]:
bb.verify.joint(world, world_history, book, settings)

### Perplexity, from the first line — and the forecast against its floor

Section 8's own headline instruction: **the number to watch is perplexity, from the first line.** This is where it lives. Two dictionaries, watched side by side — either one collapsing (the naming loss finding its way back into the vocabulary) is invisible in the loss curve, but never invisible here.

Alongside it: how well the model **draws tomorrow's candle** against the floor it must beat — **shrugging**, drawing the average candle every time.

In [ ]:
bb.plots.joint_progress(world_history);

### What the model actually predicts

The most directly readable output the whole project produces: the candle the model draws for tomorrow, beside what tomorrow actually turned out to be.

> ⚠️ **Read this honestly.** A model that has learned nothing draws a row of small, near-identical candles — the average of everything, the safest guess when you know nothing. Wide, varied predicted candles mean it is **committing** to something. Whether it commits *correctly* is a separate question — the checks above answer that, not this picture.

In [ ]:
bb.plots.predicted_candles(world, book, batches, prices, settings);

### Look at what one word remembered

Enough numbers. **Look at it.**

Take a handful of days the model has never seen, squeeze them into a single TS word, and
rebuild the candles from that word alone.

> **How can it draw a candle?** The model never sees a candle — raw prices trend and differ
> wildly between companies, and feeding them in would put drift and company-identity
> straight back. What it sees is the candle's **shape**: `gap`, `body`, `upper_wick`,
> `lower_wick` — all ratios, all scale-free. Given the previous close, those four numbers
> rebuild open/high/low/close **exactly**. So what you're looking at is precisely what the
> word remembered, not an artist's impression.

In [ ]:
bb.plots.remembered(tokenizer, batches, prices, settings);

### What it kept, and what it threw away

The reconstruction above is *smoother* than reality. That is not a failure — it is the whole point of the bottleneck, and the chart below shows exactly what happened.

One word cannot carry everything. Forced to choose, the model keeps the **slow context** — the trend, the volatility regime, the momentum — and discards the **day-to-day wiggle**.

Which is remarkable, because we deliberately *gave* it the candle. **It looked at the candle and decided the candle was noise.**

In [ ]:
bb.plots.kept_and_lost(tokenizer, batches, settings);

### ⚠️ The headline number is an average — and it is hiding something

The chart above shows a reconstruction that looks smooth and mostly right. That is **true, and misleading on its own** — the family breakdown below is what gives it away.

> ⚠️ **The table just below is a HISTORICAL illustration, not this run's numbers.** It was measured under the OLD, reconstruction-only, no-fusion tokenizer, before this branch trained everything jointly (encoders, both codebooks, the fusion and the GPT, together, against tomorrow's candle). The *pattern* it shows — keep the slow, smooth indicators, discard the actual candle — is still the shape of the finding, but the real numbers for *this* run are printed by the very next cell (`by_family`). Read those as the current answer; read the table below only as "here is what that pattern looks like".

| kept (historical) | thrown away (historical) |
|---|---|
| `macd` **85%** · `rsi` **75%** · `sma_ratio_20` **73%** | `gap` **4%** · `body` **5%** · `log_return` **9%** · `lower_wick` **−1%** |

**The token learned the trend and knows almost nothing about the actual candles.** That is exactly why the reconstruction is a smooth staircase: with the candle features that low, the decoder can only draw their *average* — and the average gap and body are slightly positive, because stocks drift up. So it produces a tidy ramp.

### And it is *right* to do that

One token is **9 bits** (`log₂ 512`). Fifteen days of MACD is a **smooth curve** — a handful of numbers describe it. Fifteen days of candle bodies are **fifteen independent random numbers** — incompressible. The token spends its nine bits on what can *actually* be compressed, and there is no way for it not to.

So: **never quote an average survival number on its own.** Break it apart — the next cell does exactly that, for real, on this run.

In [ ]:
_, by_family = bb.plots.kept_by_family(tokenizer, batches, settings)

print(by_family.sort_values(ascending=False).to_string(float_format=lambda v: f"{v:6.1%}"))

### Is CS actually working?

The rebuild score can't tell us. We've established that one word out of 512 **cannot** redraw thirty companies — nothing could. Judging CS on that is judging a fish on climbing.

So ask the question we actually care about:

> **Does the token know what the market *did* that day?**

Take every day the model has never seen, read its word, and check whether the word tells you anything about how the market really behaved. If the words separate a calm grind from a panic, CS has learned the market's moods — whatever its rebuild score says.

The number is an **R²**: knowing only which word a day got, how much of the market's behaviour can you account for?

And to stop us fooling ourselves: with 512 words and only a few hundred test days, a word could look informative **by sheer luck**. So we also score a **shuffled** assignment — the same words, handed to the wrong days. That's the "learned nothing" floor, *measured* rather than assumed.

> ⚠️ **What this does and does not show.** The token was *shown* these features, so this measures **what survived the squeeze**, not prediction. That is precisely what makes it interesting: given both, watch which one it chose to keep.

In [ ]:
moods = bb.diagnostics.market_moods(cs, batches, settings)

bb.verify.market_moods(moods)

### What the market words mean

**Look at the right-hand panel carefully — it is a trap.**

Its bars look like a pattern: some words green, some red, as if the model knew which way the market would go. **They are noise.** The score printed on the panel says so — the words explain no more of the market's *direction* than a random shuffle does.

The left panel is real, and it is not close.

> **The tokenizer was shown both, and it kept the volatility and threw the direction away.**

Which is exactly right, and it discovered it on its own. **Volatility clusters**: calm follows calm, panic follows panic — there is a regime to find. **Direction does not**: tomorrow's up-or-down is close to a coin flip, and a compressor with only 512 words to spend is not going to waste them on a coin flip.

This is the same thing TS did — *keep the regime, discard the wiggle* — now at the level of the whole market. And it is exactly what the fusion needs from CS: not a redrawing of the market, but **what kind of day it was**.

In [ ]:
bb.diagnostics.moods_plot(moods);

### The attention map — what each company chose to read

The fusion gives every company's token one question to ask the market:

> *"Of everything the market is showing me, what matters to **me**?"*

The answer is a set of weights summing to 1 — and it is **the most directly readable thing the whole model produces.** Everything else is a number you have to trust. This you can simply look at.

**The first thing to check is whether it is choosing at all.** If every weight sits at about `1/keys`, the attention is **flat**: the company is reading the whole market equally, which is the same as reading an *average* of it, which is the same as **the cross-attention not being there.**

> ⚠️ A flat map means the fusion is doing none of the work it was built for — and **that will never show up in a loss curve.**

What the map shows depends on `fusion["attend_to"]`:

| setting | the map becomes |
|---|---|
| `"days"` | which of the market's **recent days** each company read |
| `"companies"` | **which other companies** each company read — a bank attending to banks, a chip-maker to chip-makers. **A company-to-company map the model drew by itself.** |
| `"cells"` | both, shown as two marginals |

In [ ]:
# The third argument is how many TEST batches to read, not the `batches` tensors object
# above (that name is already taken) -- None reads the whole test period.
read = bb.attention.gather(world, book, None, settings)

bb.attention.plot(read);

In [ ]:
# Who does each company read most? Only meaningful when the keys ARE companies.
if settings["fusion"]["attend_to"] in ("companies", "cells"):
    display(bb.attention.neighbours(read))
else:
    print("Set  fusion['attend_to'] = 'companies'  to get a company-to-company map.")
    print("With 'days', the 5 keys are five consecutive days of the same market —")
    print("nearly identical, so there is very little for the attention to choose between.")